
# Incident Analysis With LLM

Ноутбук 
1. Загружает таблицу в формате `Incident_analysis_march.ipynb`.
2. Выполняет минимальную предобработку через `IncidentRequestsPreprocessor` без `natasha`.
3. Классифицирует каждую заявку с помощью LLM и добавляет колонку `Тип инцидента`.

Важные точки настройки находятся в ячейке `Конфиг`: путь к данным, список `COLS2DROP`, список `INCIDENT_TYPES` и `MODEL_NAME`.


In [1]:
from pathlib import Path
import json
import re
import sys

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path.cwd()
if not (ROOT / "incident_requests").exists() and (ROOT.parent / "incident_requests").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT))

from incident_requests import IncidentRequestsPreprocessor

pd.set_option("display.max_colwidth", None)

from tqdm.auto import tqdm
from huggingface_hub import snapshot_download
from huggingface_hub.utils import enable_progress_bars
enable_progress_bars()

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

In [2]:
from huggingface_hub import login
login()

In [3]:
# Конфиг
INPUT_PATH = ROOT / "research/data" / "march_incidents.xlsx"
SHEET_NAME = 0

COLS2DROP = [
    "Источник",
    "Категория",
]

TEXT_COLUMN = "Описание"
TYPE_COLUMN = "Тип инцидента"
OTHER_LABEL = "Прочее"
EMPTY_LABEL = "Не определен"

INCIDENT_TYPES = [
    "Стояк ГВС",
    "Труба ГВС",
    "Стояк ХВС",
    "Труба ХВС",
]

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE_MAP = "auto"
MAX_NEW_TOKENS = 96
OUTPUT_PATH = ROOT / "research" / "incident_analysis_llm_output.parquet"


In [4]:
def load_table(path: Path, sheet_name=0) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, sheet_name=sheet_name)

    if suffix == ".csv":
        return pd.read_csv(path)

    raise ValueError(f"Неподдерживаемый формат файла: {suffix}")


df = load_table(INPUT_PATH, sheet_name=SHEET_NAME)
processor = IncidentRequestsPreprocessor(
    df,
    columns_to_drop=COLS2DROP,
    detect_incident_type=False,
    use_natasha=False,
)
preprocessed_df = processor.preprocess()

print(preprocessed_df.dtypes)
display(preprocessed_df.head())


Дата                                 datetime64[us]
Время                                           str
Адрес                                           str
Пом.                                            str
Подкатегория                                    str
Описание                                        str
Комментарий к выполненным работам           float64
Статус заявки                                   str
Желаемое время выполнения                       str
Дата исполнения                       datetime64[s]
Исполнители                                 float64
Координаторы                                    str
Перечень материалов                         float64
Услуги                                      float64
Стоимость                                     int64
Вложения                                        str
dtype: object


,Дата,Время,Адрес,Пом.,Подкатегория,Описание,Комментарий к выполненным работам,Статус заявки,Желаемое время выполнения,Дата исполнения,Исполнители,Координаторы,Перечень материалов,Услуги,Стоимость,Вложения
0,2026-03-15,23:19,"г Санкт-Петербург, п Парголово, ул Николая Рубцова, д. 5 стр. 1",309,Протечка,"СИЛЬНАЯ ТЕЧЬ СТОЯКА ГВС В ВАННОЙ, ОТКЛ. СТ. 29 ГВС В/З, ТРЕБ. ЗАМЕНА ОТСЕЧНОГО КРАНА, ВЫРВАЛО ШТОК\nГИЛЬДИЯ: Заменили два крана 1/2 на хгвс ст. 29 в/з\nЗапустили развоздушили",NaN,Принята к исполнению,с 09:00 16.03.2026 по 13:00 16.03.2026,NaT,NaN,"ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА, Бригадир санитарно-технических работ Журавский Антон Петрович",NaN,NaN,0,Нет
1,2026-03-15,21:32,"г Санкт-Петербург, пр-кт Гладышевский, д. 38 корп. 2 стр. 1",267,Протечка,течь с потолка-ТЕЧЬ С КРОВЛИ,NaN,Принята к исполнению,с 09:00 16.03.2026 по 15:00 16.03.2026,NaT,NaN,Начальник отделения Григорьев Игорь Валерьевич,NaN,NaN,0,Да
2,2026-03-15,20:48,"г Санкт-Петербург, п Парголово, ул Фёдора Абрамова, д. 21 корп. 1 лит. А",575,Протечка,"ТЕЧЬ ПО СТЕНЕ В КОМНАТЕ +ТЕЧЕТ В КВАРТИРНОМ КОРИДОРЕ ,ТЕЧЬ СПУСКНИКА НА ТЕХ/ЭТ ,ЗАКРЫЛИ СПУСКНИК",NaN,Принята к исполнению,с 21:00 15.03.2026 по 21:15 15.03.2026,NaT,NaN,"Главный инженер Софронов Андрей Иванович, ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА",NaN,NaN,0,Нет
3,2026-03-15,20:29,"г Санкт-Петербург, п Парголово, ул Фёдора Абрамова, д. 16 корп. 1 лит. А",NaN,Лифт,3 пар. пас. 1934 застревание,NaN,Принята к исполнению,с 20:30 15.03.2026 по 19:00 16.03.2026,NaT,NaN,"Ведущий инженер по подъемно-транспортному оборудованию Гореликов Павел Вячеславович, Инженер по подъемно-транспортному оборудованию Горюнов Сергей Александрович, Спецтрест 27 – ЛИФТ Шкапцов А Л",NaN,NaN,0,Нет
4,2026-03-15,19:47,"г Санкт-Петербург, п Парголово, ул Валерия Гаврилина, д. 3 корп. 1 лит. А",21,Протечка,"течь п/суш. (981) 822-05-22, течь соед. на ст. гвс н/з, откл. ст. гвс 18, ТРЕБ. РАЗБОР КОРОБА В КВ. 21 - РАЗОБРАЛИ, ЖДУТ 16.03.2026 В 10-00",NaN,Принята к исполнению,с 10:00 16.03.2026 по 13:00 16.03.2026,NaT,NaN,ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА,NaN,NaN,0,Нет


In [5]:

def build_classification_prompt(
    description: str,
    incident_types: list[str],
    other_label: str = OTHER_LABEL,
) -> str:
    labels = [*incident_types, other_label]
    options = "\n".join(f"- {label}" for label in labels)

    return f"""Определи тип инцидента по тексту заявки.
Выбери ровно один вариант из списка.
Если ни один вариант не подходит, выбери "{other_label}".

Доступные типы инцидентов:
{options}

Текст заявки:
{description}

Верни только JSON без пояснений:
{{"incident_type": "<один вариант из списка>"}}"""


def load_generation_model(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    cache_path = snapshot_download(model_name, resume_download=True)
    
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        cache_path,
        dtype=torch_dtype,
        device_map=DEVICE_MAP,
        trust_remote_code=True,
    )
    model.eval()
    return tokenizer, model


In [6]:

def parse_model_response(response_text: str, labels: list[str], other_label: str) -> str:
    match = re.search(r"\{.*\}", response_text, flags=re.S)
    if match:
        try:
            payload = json.loads(match.group(0))
            label = str(payload.get("incident_type", "")).strip()
            if label in labels:
                return label
        except json.JSONDecodeError:
            pass

    cleaned_text = response_text.strip().strip('"')
    if cleaned_text in labels:
        return cleaned_text

    for label in labels:
        if label.lower() in cleaned_text.lower():
            return label

    return other_label


def generate_incident_type(
    description: str,
    tokenizer,
    model,
    incident_types: list[str],
    other_label: str = OTHER_LABEL,
    empty_label: str = EMPTY_LABEL,
) -> str:
    if pd.isna(description) or not str(description).strip():
        return empty_label

    labels = [*incident_types, other_label]
    messages = [
        {
            "role": "system",
            "content": "Ты размечаешь заявки ЖКХ и возвращаешь только валидный JSON.",
        },
        {
            "role": "user",
            "content": build_classification_prompt(str(description), incident_types, other_label),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = generated[0][model_inputs["input_ids"].shape[1]:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return parse_model_response(response_text, labels, other_label)


def classify_incidents(
    dataframe: pd.DataFrame,
    tokenizer,
    model,
    text_column: str,
    incident_types: list[str],
) -> pd.DataFrame:
    if text_column not in dataframe.columns:
        raise ValueError(f"Колонка '{text_column}' не найдена. Есть: {list(dataframe.columns)}")

    result_df = dataframe.copy()
    result_df[TYPE_COLUMN] = [
        generate_incident_type(description, tokenizer, model, incident_types)
        for description in tqdm(result_df[text_column], total=len(result_df))
    ]
    return result_df


In [7]:

tokenizer, model = load_generation_model(MODEL_NAME)
result_df = classify_incidents(
    preprocessed_df,
    tokenizer=tokenizer,
    model=model,
    text_column=TEXT_COLUMN,
    incident_types=INCIDENT_TYPES,
)

display(result_df[[TEXT_COLUMN, TYPE_COLUMN]].head())
display(
    result_df[TYPE_COLUMN]
    .value_counts(dropna=False)
    .rename_axis(TYPE_COLUMN)
    .reset_index(name="count")
)


/home/fedor/Projects/building_maintenance_agents/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  0%|          | 0/458 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

In [ ]:

if OUTPUT_PATH.suffix == ".csv":
    result_df.to_csv(OUTPUT_PATH, index=False)
elif OUTPUT_PATH.suffix in {".xlsx", ".xls"}:
    result_df.to_excel(OUTPUT_PATH, index=False)
else:
    result_df.to_parquet(OUTPUT_PATH, index=False)

OUTPUT_PATH
